<a href="https://colab.research.google.com/github/usma-stats/ma206x/blob/main/ma206x_ay27_1_teaching_slate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pulp

# ----------------------------
# BASIC SETS
# ----------------------------

n_instructors = 13
instructors = range(n_instructors)

days = [1, 2]
hours = ['A','B','C','D','E','F']
hour_index = {'A':1,'B':2,'C':3,'D':4,'E':5,'F':6}

# ----------------------------
# REAL SECTION DISTRIBUTION
# ----------------------------

section_counts = {
    (1,'A'): 6,
    (1,'B'): 2,
    (1,'C'): 2,
    (1,'D'): 3,
    (1,'E'): 3,
    (1,'F'): 2,
    (2,'A'): 6,
    (2,'B'): 2,
    (2,'C'): 3,
    (2,'D'): 3,
    (2,'E'): 4,
    (2,'F'): 4
}

section_day = {}
section_hour = {}

s = 0
for (d,h), count in section_counts.items():
    for _ in range(count):
        section_day[s] = d
        section_hour[s] = h
        s += 1

sections = range(s)  # should be 40

# ----------------------------
# LOAD STRUCTURE
# ----------------------------

full_load = list(range(7))       # 7 instructors teach 4
light_load = list(range(7,13))   # 6 instructors teach 2

target_load = {}

for i in full_load:
    target_load[i] = 4
for i in light_load:
    target_load[i] = 2

# ----------------------------
# MODEL
# ----------------------------

model = pulp.LpProblem("Instructor_Scheduling", pulp.LpMinimize)

# Assignment variable
x = pulp.LpVariable.dicts(
    "assign",
    ((i,s) for i in instructors for s in sections),
    cat="Binary"
)

# ----------------------------
# HARD CONSTRAINTS
# ----------------------------

# 1) Each section assigned exactly once
for s in sections:
    model += pulp.lpSum(x[(i,s)] for i in instructors) == 1

# 2) Exact instructor load
for i in instructors:
    model += pulp.lpSum(x[(i,s)] for s in sections) == target_load[i]

# 3) No time conflicts
for i in instructors:
    for d in days:
        for h in hours:
            model += (
                pulp.lpSum(
                    x[(i,s)]
                    for s in sections
                    if section_day[s] == d and section_hour[s] == h
                ) <= 1
            )

# ----------------------------
# TEACH HOUR INDICATOR
# ----------------------------

y = pulp.LpVariable.dicts(
    "teach_hour",
    ((i,d,h) for i in instructors for d in days for h in hours),
    cat="Binary"
)

for i in instructors:
    for d in days:
        for h in hours:
            model += (
                y[(i,d,h)] ==
                pulp.lpSum(
                    x[(i,s)]
                    for s in sections
                    if section_day[s] == d and section_hour[s] == h
                )
            )

# ----------------------------
# MIN / MAX HOUR (SPAN)
# ----------------------------

min_hour = pulp.LpVariable.dicts(
    "min_hour",
    ((i,d) for i in instructors for d in days),
    lowBound=1, upBound=6
)

max_hour = pulp.LpVariable.dicts(
    "max_hour",
    ((i,d) for i in instructors for d in days),
    lowBound=1, upBound=6
)

for i in instructors:
    for d in days:
        for h in hours:
            h_val = hour_index[h]

            # If teaching at h → min_hour ≤ h
            model += min_hour[(i,d)] <= h_val + (1 - y[(i,d,h)]) * 6

            # If teaching at h → max_hour ≥ h
            model += max_hour[(i,d)] >= h_val - (1 - y[(i,d,h)]) * 6

# Span definition
span = pulp.LpVariable.dicts(
    "span",
    ((i,d) for i in instructors for d in days),
    lowBound=0
)

for i in instructors:
    for d in days:
        model += span[(i,d)] == max_hour[(i,d)] - min_hour[(i,d)]

# ----------------------------
# GAP PENALTY
# ----------------------------

gap = pulp.LpVariable.dicts(
    "gap",
    ((i,d,h) for i in instructors for d in days for h in hours[:-2]),
    cat="Binary"
)

for i in instructors:
    for d in days:
        for idx in range(len(hours) - 2):
            h1 = hours[idx]
            h2 = hours[idx+1]
            h3 = hours[idx+2]

            model += gap[(i,d,h1)] >= (
                y[(i,d,h1)] + y[(i,d,h3)] - y[(i,d,h2)] - 1
            )

# ----------------------------
# OBJECTIVE
# ----------------------------

alpha = 1.0
beta = 2.0

model += (
    alpha * pulp.lpSum(span[(i,d)] for i in instructors for d in days)
    +
    beta * pulp.lpSum(
        gap[(i,d,h)]
        for i in instructors
        for d in days
        for h in hours[:-2]
    )
)

# ----------------------------
# SOLVE
# ----------------------------

model.solve(pulp.PULP_CBC_CMD(msg=True))

print("Status:", pulp.LpStatus[model.status])

# ----------------------------
# PRINT CLEAN SCHEDULE
# ----------------------------

for i in instructors:
    print(f"\nInstructor {i}:")
    for d in days:
        taught = [
            h for h in hours
            if y[(i,d,h)].value() == 1
        ]
        print(f"  Day {d}: {taught}")
